# Molecular docking -- Lamium_amplexicaule vs Aldose Reductase (2FZD)



In [1]:
from pymol import cmd
import py3Dmol

from vina import Vina

from openbabel import pybel

from rdkit import Chem
from rdkit.Chem import AllChem, Draw

from meeko import MoleculePreparation

import MDAnalysis as mda
from MDAnalysis.coordinates import PDB

import prolif as plf
from prolif.plotting.network import LigNetwork


import sys, os
sys.path.insert(1, "/home/ali/medical plants/docking/Jupyter_Dock-main/utilities/")
from utils import fix_protein, getbox, generate_ledock_file, pdbqt_to_sdf, dok_to_sdf


import warnings
warnings.filterwarnings("ignore")
%config Completer.use_jedi = False

/home/ali/Downloads/or/envs/jupdock_clean/lib/python3.10/site-packages/MDAnalysis/topology/tables.py:52: DeprecationWarning: Deprecated in version 2.8.0
MDAnalysis.topology.tables has been moved to MDAnalysis.guesser.tables. This import point will be removed in MDAnalysis version 3.0.0
  warnings.warn(wmsg, category=DeprecationWarning)


It's a good idea to run the test protocols before attempting custom projects. Later, the user can specify the location of his/her project and save all of the files separately.

In [2]:
os.chdir('/home/ali/Desktop/Algerian_Lamiaceae_v2/results/docking_results/')


## 1. Feching system and cleanup

Implementing Pymol is a simple way to download PDB structures. The user can launch this or any other Jupyter Dock's protocol by providing his or her own files.

In [3]:
cmd.fetch(code='2FZD',type='pdb1')
cmd.select(name='Prot',selection='polymer.protein')
cmd.save(filename='2FZD_clean.pdb',format='pdb',selection='Prot',state=1)

# 2FZD's real inhibitor is tolrestat. Auto-detect rather than hardcoding a
# guessed code: exclude the required NADP+ cofactor (NAP) and water, and
# restrict to chain A -- this structure's own publication notes that some
# tolrestat-aldose-reductase crystal forms picked up EXTRA, non-biological
# copies from crystal packing effects (up to 4 in one related structure),
# so we deliberately avoid grabbing more than one copy.
cmd.select(name='NativeLig', selection='organic and not resn NAP+NDP+HOH and chain A')
_natlig_model = cmd.get_model('NativeLig', state=1)
_natlig_resns = sorted(set((a.resn, a.resi) for a in _natlig_model.atom))
print("Native ligand candidate(s) found (resn, resi):", _natlig_resns)
print("Total atom count:", len(_natlig_model.atom))

# If more than one distinct (resn, resi) pair appears above, narrow the
# selection to a single one, e.g.:
# cmd.select(name='NativeLig', selection="resn XXX and resi YYY and chain A")
cmd.save(filename='2FZD_native_lig.mol2', format='mol2', selection='NativeLig', state=1)
cmd.delete('all')


Native ligand candidate(s) found (resn, resi): [('TOL', '317')]
Total atom count: 24


## 2. Protein and ligand sanitization

### 2.1. Protein Sanitization

#### Method 1: LePro

The Lephar molecular docking suite includes a very powerful tool for automatically preparing proteins for molecular docking. As a result, for these protocols, it will be the preferred tool for protein preparation. The protein prepared by LePro can be used in AutoDock VIna and LeDock.

In [4]:
!"/home/ali/medical plants/docking/Jupyter_Dock-main/bin/lepro_linux_x86" # Launch this cell to see parameters


************************************************************
*      LePro                                               *
*            Add hydrogen atoms to a protein &             *
*            write the input file for LeDock               *
*      Copyright (C) 2013-21 Hongtao Zhao, PhD             *
*      Email: htzhaovv@gmail.com                           *
************************************************************
----------Usage:                                                                       
          lepro [PDB file] [-rot || -metal || -p]                                        
          -rot  [[chain] resid] align principal axes of the binding site with Cartesian
          -metal keep ZN/MN/CA/MG                                                      
          -metal -p redistribute metal charge to protein                               



In [5]:
!"/home/ali/medical plants/docking/Jupyter_Dock-main/bin/lepro_linux_x86" {'2FZD_clean.pdb'}

os.rename('pro.pdb','2FZD_clean_H.pdb')


#### Method 2: fix_protein (PDBFixer)

For proteins with missing amino acids or residues, or to ensure a more thorough sanitization of protein Jupyter Dock includes the _fix_protein()_ function, which employs PDBFixer to correct a wide range of common errors in protein pdb files. Furthermore, PDBFixer enables the assignment of pH-dependent protonation states to proteins.


>**Warning 1:**  It is possible to encounter problems with protein fixing when using **_fix_protein()_** and AutoDock Tools' **_prepare_receptor_** or when running LeDock. To resolve the issue, it is best to set the parameter _-A hydrogens_ in _prepare receptor_.


>**Hint:** PDBFixer is a great solution for many systems because it can solve serious problems in PDB files. As a result, PDBFixer renumbers the residues beginning with 1 regardless of the numbering on the original PDB file. To address this issue, the **_fix_protein()_** function includes a protocol for atomically renumbering the residues in accordance with the original PDB file.


**_fix_protein ( params )_**

Params:
      
  - **filename**: _str or path-like_ ; input file containing protein struture to be modified, file extrension must be pdb

  - **addHs_pH**: _float_ ; Add hydrogens at user defined pH

  - **try_renumberResidues**: _bool_ ; By default PDBFixer renumarets residues starting in 1, this option tries to recover originar residues numbering
    
  - **output**: _str or path-like_ ; output filename, extension must be pdb

```
fix_protein(filename='2FZD_clean.pdb',addHs_pH=7.4,try_renumberResidues=True,output='2FZD_clean_H.pdb')
```

## 4. Docking with AutoDock Vina

AutoDock Vina (Vina) is one of the docking engines in the AutoDock Suite, together with AutoDock4 (AD4), AutoDockGPU, AutoDockFR, and AutoDock-CrankPep, and arguably among the most widely used and successful docking engines. The reasons for this success are mostly due to its ease of use and its speed (up to 100x faster than AD4), when compared to the other docking engines in the suite and elsewhere, as well as being open source.

### 4.1. Protein preparation

After sanitization, the protein docking preparation includes converting it to the PDBQT file format, which stores the atomic coordinates, partial charges, and AutoDock atom types for both the receptor and the ligand. 

>**Hint:** Despite the fact that the PDBQT format includes changes (Q) and atom types (T) for molecular docking. The charges are not required for Autodock Vina, which computes electrostatic interactions using its own force field. Although, when using AutoDock instead of AutoDock Vina, the Q term is required. More information can be found here:https://autodock-vina.readthedocs.io/en/latest/introduction.html

#### Method 1: AutoDock Tools prepare_receptor

The AutoDock Tools are the best way to prepare the receptor for AutoDock Vina. Nonetheless, AutoDock Tools is a comprehensive suite of programs and scripts that are difficult to manage. Jupyter Dock execute the **_prepare receptor_** and **_prepare_ligand_** functions on their own. As a result, obtaining proper PDBQT files for AutoDock Vina has never been easier.

>**Warning:** There is currently only one method for preparing a PDBQT file for a receptor. It is expected that new methods apart from AutoDock Tools executables will be included in the near future into Jupyter Dock.

In [6]:
!"/home/ali/medical plants/docking/Jupyter_Dock-main/bin/prepare_receptor" #Lauch this cell to see parameters


prepare_receptor4: receptor filename must be specified.
Usage: prepare_receptor4.py -r filename

    Description of command...
         -r   receptor_filename 
        supported file types include pdb,mol2,pdbq,pdbqs,pdbqt, possibly pqr,cif
    Optional parameters:
        [-v]  verbose output (default is minimal output)
        [-o pdbqt_filename]  (default is 'molecule_name.pdbqt')
        [-A]  type(s) of repairs to make: 
             'bonds_hydrogens': build bonds and add hydrogens 
             'bonds': build a single bond from each atom with no bonds to its closest neighbor
             'hydrogens': add hydrogens
             'checkhydrogens': add hydrogens only if there are none already
             'None': do not make any repairs 
             (default is 'None')
        [-C]  preserve all input charges ie do not add new charges 
             (default is addition of gasteiger charges)
        [-p]  preserve input charges on specific atom types, eg -p Zn -p Fe
        [-U]  cle

In [7]:
!"/home/ali/medical plants/docking/Jupyter_Dock-main/bin/prepare_receptor" -v -r {'2FZD_clean_H.pdb'} -o {'2FZD_clean_H.pdbqt'}

# If this raises "UnboundLocalError: local variable 'bonds' referenced before
# assignment", use this fallback:
# import subprocess
# r = subprocess.run(["obabel", "2FZD_clean_H.pdb", "-O", "2FZD_clean_H.pdbqt", "-xr"],
#                     capture_output=True, text=True)
# print(r.stdout, r.stderr)
#
# Note: prepare_receptor may also warn about the NADP+ cofactor if any part
# of it remained in the file (it shouldn't, since only 'Prot' -- polymer
# atoms -- was saved to 2FZD_clean.pdb above). If AR's activity genuinely
# requires NADP+ to be present for realistic docking (it's a cofactor-
# dependent oxidoreductase), that's a modeling limitation of rigid-receptor
# small-molecule docking generally, not specific to this pipeline.


set verbose to  True
set receptor_filename to  2FZD_clean_H.pdb
set outputfilename to  2FZD_clean_H.pdbqt
read  2FZD_clean_H.pdb
setting up RPO with mode= automatic and outputfilename=  2FZD_clean_H.pdbqt
charges_to_add= gasteiger
delete_single_nonstd_residues= None
adding gasteiger charges to peptide


In [8]:
fix_protein(filename='2FZD_clean.pdb',addHs_pH=7.4,try_renumberResidues=True,output='2FZD_clean_H_fix.pdb')

## 7. Screening PHYTO compounds (Lamium_amplexicaule) against Aldose Reductase (2FZD)

Reuses the receptor (`2FZD_clean_H.pdb`/`.pdbqt`) and the docking box
(native ligand tolrestat, chain A, `state=1`) prepared above. Each
compound docked in its own cell with its own uniquely-named variables,
seed pinned at 42. Source: Laggoune et al. 2008.

**Chrysoeriol is a flavone** -- the AR literature consistently shows
flavonoids scoring notably stronger (-7.5 to -9.9 kcal/mol) than the
monoterpenes tested in every previous round of this project (-5.8 to -6.8
kcal/mol, with beta-Caryophyllene's -7.99 the sole exception). This is the
first direct test of whether that class-level pattern holds for your own
data.

**Flag for Stigmasterol**: given its history in this project (failed
against COX-2/5IKQ until redocked against 3NTG; failed against 5-LOX/3O8Y
with the default box until widened), the same widened-box/higher-
exhaustiveness fallback is included below in case the default result
looks weak or the poses collapse after 1-2.

In [9]:
import os
from rdkit import Chem
from rdkit.Chem import AllChem, rdMolDescriptors
from openbabel import openbabel as ob
from vina import Vina
import pandas as pd
import matplotlib.pyplot as plt

DOCKING_DIR = "/home/ali/Desktop/Algerian_Lamiaceae_v2/results/docking_results/"
FIGURES_DIR = os.path.join(DOCKING_DIR, "figures")
os.makedirs(FIGURES_DIR, exist_ok=True)
os.chdir(DOCKING_DIR)

RECEPTOR_PDB   = os.path.join(DOCKING_DIR, "2FZD_clean_H.pdb")
RECEPTOR_PDBQT = os.path.join(DOCKING_DIR, "2FZD_clean_H.pdbqt")
NATIVE_LIG     = os.path.join(DOCKING_DIR, "2FZD_native_lig.mol2")

assert os.path.exists(RECEPTOR_PDBQT), "Run the receptor prep cells above first."
assert os.path.exists(NATIVE_LIG), "Run the native ligand extraction cell above first."

VINA_SEED = 42  # pinned for reproducibility

def prepare_ligand_from_smiles(name, smiles, n_confs=20):
    """SMILES -> multiple 3D conformers (ETKDGv3) -> MMFF94 minimize each ->
    keep lowest-energy conformer -> PDB -> PDBQT (AutoDockTools prepare_ligand,
    with OpenBabel fallback)."""
    import subprocess
    m = Chem.MolFromSmiles(smiles)
    m = Chem.AddHs(m)

    params = AllChem.ETKDGv3()
    params.randomSeed = 42
    conf_ids = AllChem.EmbedMultipleConfs(m, numConfs=n_confs, params=params)

    energies = []
    for cid in conf_ids:
        ff = AllChem.MMFFGetMoleculeForceField(m, AllChem.MMFFGetMoleculeProperties(m), confId=cid)
        if ff is None:
            continue
        ff.Minimize(maxIts=2000)
        energies.append((ff.CalcEnergy(), cid))
    if not energies:
        for cid in conf_ids:
            ff = AllChem.UFFGetMoleculeForceField(m, confId=cid)
            ff.Minimize(maxIts=2000)
            energies.append((ff.CalcEnergy(), cid))

    best_energy, best_cid = min(energies, key=lambda x: x[0])
    print(f"  {name}: {len(conf_ids)} conformers generated, "
          f"lowest energy = {best_energy:.2f} kcal/mol (conf {best_cid})")

    pdb_path   = os.path.join(DOCKING_DIR, f'_lig_{name}.pdb')
    pdbqt_path = os.path.join(DOCKING_DIR, f'_lig_{name}.pdbqt')
    Chem.MolToPDBFile(m, pdb_path, confId=best_cid)

    PREP_LIG_BIN = "/home/ali/medical plants/docking/Jupyter_Dock-main/bin/prepare_ligand"
    result = subprocess.run([PREP_LIG_BIN, "-v", "-l", pdb_path, "-o", pdbqt_path],
                             capture_output=True, text=True)
    if not os.path.exists(pdbqt_path) or os.path.getsize(pdbqt_path) == 0:
        obConv = ob.OBConversion()
        obConv.SetInAndOutFormats('pdb', 'pdbqt')
        obmol = ob.OBMol()
        obConv.ReadFile(obmol, pdb_path)
        obConv.WriteFile(obmol, pdbqt_path)

    return pdbqt_path


### 7.1. Recompute the docking box (native ligand tolrestat, `state=1`)

In [10]:
cmd.load(filename=RECEPTOR_PDB, format='pdb', object='prot', state=1)
cmd.load(filename=NATIVE_LIG, format='mol2', object='natlig', state=1)

center, size = getbox(selection='natlig', extending=5.0, software='vina')
cmd.delete('all')

print(center, '\n', size)


{'center_x': 16.690499782562256, 'center_y': -7.611000299453735, 'center_z': 13.86899995803833} 
 {'size_x': 13.606999397277832, 'size_y': 15.78000020980835, 'size_z': 19.872000694274902}


### 7.1b. Reference-ligand redocking validation

2FZD (aldose reductase) is **not** a metalloenzyme -- its cofactor is NADP+ (excluded from the native-ligand selection above), not a catalytic metal. A `RMSD = None` result here means the reference molecule failed to parse with correct bond orders (RDKit's `sanitize=False` mol2 fallback), not a scoring-function limitation -- so this does **not** need the AD4Zn treatment used for ACE. Fixed below by parsing the reference via OpenBabel instead.

In [11]:
# ---- Reference-ligand redocking validation (reviewer Comment 12) ----
# 2FZD (aldose reductase) is NOT a metalloenzyme -- its cofactor is NADP+
# (explicitly excluded above), not a catalytic metal ion, so plain Vina's
# lack of a metal-coordination term is NOT the issue here (unlike ACE/1O86).
# ref_mol is parsed via OpenBabel (mol2 -> sdf) rather than RDKit's
# MolFromMol2File(sanitize=False), which silently produces a garbage-bonded
# reference (all-bracket atoms, no aromaticity) and causes RMSD=None --
# this was the actual root cause for ACE too, before zinc was even relevant.
import subprocess
from rdkit.Chem.rdMolAlign import GetBestRMS
from rdkit.Chem import AllChem
from openbabel import openbabel as ob

REDOCK_DIR = os.path.join(DOCKING_DIR, "redocking_validation")
os.makedirs(REDOCK_DIR, exist_ok=True)

PDB_CODE = os.path.basename(RECEPTOR_PDB).split('_')[0]

# 1) Prepare the NATIVE ligand (tolrestat) for docking -- from its crystal
#    geometry (NATIVE_LIG, the mol2 extracted from the PDB), NOT a fresh
#    SMILES conformer.
native_lig_pdbqt = os.path.join(REDOCK_DIR, f"{PDB_CODE}_native_lig_redock.pdbqt")
PREP_LIG_BIN = "/home/ali/medical plants/docking/Jupyter_Dock-main/bin/prepare_ligand"
result = subprocess.run([PREP_LIG_BIN, "-v", "-l", NATIVE_LIG, "-o", native_lig_pdbqt],
                         capture_output=True, text=True)
if not os.path.exists(native_lig_pdbqt) or os.path.getsize(native_lig_pdbqt) == 0:
    obConv = ob.OBConversion()
    obConv.SetInAndOutFormats('mol2', 'pdbqt')
    obmol = ob.OBMol()
    obConv.ReadFile(obmol, NATIVE_LIG)
    obConv.WriteFile(obmol, native_lig_pdbqt)

# 2) Redock into the SAME box already computed for compound screening (7.1)
v_redock = Vina(sf_name='vina', seed=VINA_SEED)
v_redock.set_receptor(RECEPTOR_PDBQT)
v_redock.set_ligand_from_file(native_lig_pdbqt)
v_redock.compute_vina_maps(center=[center['center_x'], center['center_y'], center['center_z']],
                            box_size=[size['size_x'], size['size_y'], size['size_z']])
v_redock.dock(exhaustiveness=10, n_poses=10)

redock_out_pdbqt = os.path.join(REDOCK_DIR, f"{PDB_CODE}_native_lig_redock_out.pdbqt")
v_redock.write_poses(redock_out_pdbqt, n_poses=10, overwrite=True)
redock_energies = v_redock.energies(n_poses=10)
redock_best_score = round(float(redock_energies[0][0]), 3)
print(f"Redocking best Vina score: {redock_best_score} kcal/mol")

redock_out_sdf = redock_out_pdbqt.replace('.pdbqt', '.sdf')
pdbqt_to_sdf(pdbqt_file=redock_out_pdbqt, output=redock_out_sdf)

# 3) Reference molecule -- parsed via OpenBabel (mol2 -> sdf) for correct
#    bond orders/aromaticity, instead of RDKit's mol2 reader.
native_lig_sdf = os.path.join(REDOCK_DIR, f"{PDB_CODE}_native_lig_ref.sdf")
obConv = ob.OBConversion()
obConv.SetInAndOutFormats('mol2', 'sdf')
obmol = ob.OBMol()
obConv.ReadFile(obmol, NATIVE_LIG)
obConv.WriteFile(obmol, native_lig_sdf)
ref_mol = Chem.SDMolSupplier(native_lig_sdf, removeHs=True)[0]

# 4) RMSD of each docked pose vs the ORIGINAL crystallographic pose
#    (heavy atoms only). Bond orders on each pose are reassigned from
#    ref_mol's template before comparison, since PDBQT carries none.
poses = Chem.SDMolSupplier(redock_out_sdf, removeHs=True)
rmsd_rows = []
for i, pose_mol in enumerate(poses):
    if pose_mol is None:
        rmsd_rows.append({"Pose": i + 1, "Vina_score_kcal_mol": float(redock_energies[i][0]),
                           "RMSD_vs_crystal_pose_A": None})
        continue
    try:
        pose_fixed = AllChem.AssignBondOrdersFromTemplate(ref_mol, pose_mol)
        rmsd = round(GetBestRMS(pose_fixed, ref_mol), 3)
    except Exception as e:
        rmsd = None  # topology/substructure match failed -- inspect this pose manually
    rmsd_rows.append({
        "Pose": i + 1,
        "Vina_score_kcal_mol": float(redock_energies[i][0]),
        "RMSD_vs_crystal_pose_A": rmsd,
    })

redock_df = pd.DataFrame(rmsd_rows)
print(f"=== Reference-ligand redocking validation: {PDB_CODE} / native ligand {NATIVE_LIG} ===")
display(redock_df)
best_pose_rmsd = redock_df.loc[redock_df['Vina_score_kcal_mol'].idxmin(), 'RMSD_vs_crystal_pose_A']
print(f"Best-scoring pose RMSD vs crystal structure: {best_pose_rmsd} Å")
print("Standard success criterion in the docking-validation literature: RMSD < 2.0 A")
if best_pose_rmsd is not None and best_pose_rmsd < 2.0:
    print("Redocking validation result: PASS")
else:
    print("Redocking validation result: FAIL")
    print("  -> RMSD >= 2.0 A (or could not be computed): inspect the pose visually")
    print("     before relying on this receptor/box/protocol for compound screening.")

redock_df.to_csv(os.path.join(REDOCK_DIR, f"{PDB_CODE}_redocking_validation.csv"), index=False)


[RDKit] ERROR:[09:54:04] Explicit valence for atom # 1 N, 4, is greater than permitted
[RDKit] ERROR:[09:54:04] ERROR: Could not sanitize molecule ending on line 55
[RDKit] ERROR:[09:54:04] ERROR: Explicit valence for atom # 1 N, 4, is greater than permitted


Computing Vina grid ... done.
Performing docking (random seed: 42) ... 
0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************
Redocking best Vina score: -9.24 kcal/mol
=== Reference-ligand redocking validation: 2FZD / native ligand /home/ali/Desktop/Algerian_Lamiaceae_v2/results/docking_results/2FZD_native_lig.mol2 ===


,Pose,Vina_score_kcal_mol,RMSD_vs_crystal_pose_A
0,1,-9.240,None
1,2,-7.760,None
2,3,-7.628,None
3,4,-7.617,None
4,5,-7.483,None
5,6,-7.475,None
6,7,-7.463,None
7,8,-7.308,None
8,9,-7.058,None
9,10,-7.010,None


Best-scoring pose RMSD vs crystal structure: None Å
Standard success criterion in the docking-validation literature: RMSD < 2.0 A
Redocking validation result: FAIL
  -> RMSD >= 2.0 A (or could not be computed): inspect the pose visually
     before relying on this receptor/box/protocol for compound screening.


In [12]:
print("ref_mol is None:", ref_mol is None)
print("pose0 is None:", poses[0] is None)

# Check the intermediate SDF file OpenBabel actually wrote
print("native_lig_sdf exists:", os.path.exists(native_lig_sdf))
print("native_lig_sdf size (bytes):", os.path.getsize(native_lig_sdf) if os.path.exists(native_lig_sdf) else "N/A")
print("--- native_lig_sdf content ---")
print(open(native_lig_sdf).read())

ref_mol is None: True
pose0 is None: False
native_lig_sdf exists: True
native_lig_sdf size (bytes): 2329
--- native_lig_sdf content ---
2FZD
 OpenBabel10202609543D

 24 25  0  0  0  0  0  0  0  0999 V2000
   17.1870   -8.2640    9.9370 C   0  0  0  0  0  0  0  0  0  0  0  0
   17.0300   -7.5560   16.7640 N   0  0  0  0  0  4  0  0  0  0  0  0
   15.6960   -5.8960   10.2920 O   0  0  0  0  0  0  0  0  0  0  0  0
   14.9180   -9.1750   16.5950 S   0  0  0  0  0  0  0  0  0  0  0  0
   16.7180   -7.7760   11.2690 C   0  0  0  0  0  0  0  0  0  0  0  0
   14.9120   -5.5140   18.8050 O   0  5  0  0  0  0  0  0  0  0  0  0
   15.9740   -6.6000   11.3940 C   0  0  0  0  0  0  0  0  0  0  0  0
   15.2750   -5.5640   16.6410 O   0  0  0  0  0  0  0  0  0  0  0  0
   14.8870   -4.7210   10.4690 C   0  0  0  0  0  0  0  0  0  0  0  0
   15.5430   -6.2000   12.6620 C   0  0  0  0  0  0  0  0  0  0  0  0
   15.8880   -6.9440   13.7500 C   0  0  0  0  0  0  0  0  0  0  0  0
   16.6040   -8.1460   13

### 7.2. Dock **Chrysoeriol** against Aldose Reductase (2FZD)

In [13]:
# ---- Chrysoeriol (Lamium_amplexicaule, Laggoune et al. 2008) ----
smiles_Chrysoeriol = "COC1=C(C=CC(=C1)C2=CC(=O)C3=C(C=C(C=C3O2)O)O)O"
lig_pdbqt_Chrysoeriol = prepare_ligand_from_smiles("Chrysoeriol", smiles_Chrysoeriol)

v_Chrysoeriol = Vina(sf_name='vina', seed=VINA_SEED)
v_Chrysoeriol.set_receptor(RECEPTOR_PDBQT)
v_Chrysoeriol.set_ligand_from_file(lig_pdbqt_Chrysoeriol)
v_Chrysoeriol.compute_vina_maps(center=[center['center_x'], center['center_y'], center['center_z']],
                     box_size=[size['size_x'], size['size_y'], size['size_z']])

v_Chrysoeriol.dock(exhaustiveness=10, n_poses=10)
vina_out_pdbqt_Chrysoeriol = os.path.join(DOCKING_DIR, "Chrysoeriol_2FZD_vina_out.pdbqt")
v_Chrysoeriol.write_poses(vina_out_pdbqt_Chrysoeriol, n_poses=10, overwrite=True)

energies_Chrysoeriol = v_Chrysoeriol.energies(n_poses=10)
best_score_Chrysoeriol = round(float(energies_Chrysoeriol[0][0]), 3)

print("=== Chrysoeriol ===")
print(f"Best Vina score: {best_score_Chrysoeriol} kcal/mol")
for i, e in enumerate(energies_Chrysoeriol):
    print(f"  Pose {i+1}: {e[0]:.3f}")

vina_out_sdf_Chrysoeriol = vina_out_pdbqt_Chrysoeriol.replace('.pdbqt', '.sdf')
pdbqt_to_sdf(pdbqt_file=vina_out_pdbqt_Chrysoeriol, output=vina_out_sdf_Chrysoeriol)


  Chrysoeriol: 20 conformers generated, lowest energy = 31.92 kcal/mol (conf 3)

mode |   affinity | dist from best mode
     | (kcal/mol) | rmsd l.b.| rmsd u.b.
-----+------------+----------+----------
   1        -9.24          0          0
   2        -7.76      1.946      3.403
   3       -7.628      1.952      3.585
   4       -7.617      4.267       6.46
   5       -7.483      4.449      6.812
   6       -7.475      1.804      2.377
   7       -7.463        4.5      5.855
   8       -7.308      4.243      6.195
   9       -7.058      4.715      7.304
  10        -7.01      4.332      6.277
Computing Vina grid ... === Chrysoeriol ===
Best Vina score: -8.149 kcal/mol
  Pose 1: -8.149
  Pose 2: -8.008
  Pose 3: -7.942
  Pose 4: -7.522
  Pose 5: -7.472
  Pose 6: -7.462
  Pose 7: -7.404
  Pose 8: -7.240
  Pose 9: -7.233
  Pose 10: -7.100
done.
Performing docking (random seed: 42) ... 
0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|

### 7.3. Dock **Stigmasterol** against Aldose Reductase (2FZD)

Watch the full pose list for the "collapses after pose 1-2" pattern seen
previously. If it appears, uncomment the widened-box / higher-
exhaustiveness block below.

In [14]:
# ---- Stigmasterol (Lamium_amplexicaule, Laggoune et al. 2008) ----
smiles_Stigmasterol = "CC[C@H](/C=C/[C@@H](C)[C@H]1CC[C@@H]2[C@@]1(CC[C@H]3[C@H]2CC=C4[C@@]3(CC[C@@H](C4)O)C)C)C(C)C"
lig_pdbqt_Stigmasterol = prepare_ligand_from_smiles("Stigmasterol", smiles_Stigmasterol)

v_Stigmasterol = Vina(sf_name='vina', seed=VINA_SEED)
v_Stigmasterol.set_receptor(RECEPTOR_PDBQT)
v_Stigmasterol.set_ligand_from_file(lig_pdbqt_Stigmasterol)
v_Stigmasterol.compute_vina_maps(center=[center['center_x'], center['center_y'], center['center_z']],
                     box_size=[size['size_x'], size['size_y'], size['size_z']])

v_Stigmasterol.dock(exhaustiveness=10, n_poses=10)
vina_out_pdbqt_Stigmasterol = os.path.join(DOCKING_DIR, "Stigmasterol_2FZD_vina_out.pdbqt")
v_Stigmasterol.write_poses(vina_out_pdbqt_Stigmasterol, n_poses=10, overwrite=True)

energies_Stigmasterol = v_Stigmasterol.energies(n_poses=10)
best_score_Stigmasterol = round(float(energies_Stigmasterol[0][0]), 3)

print("=== Stigmasterol ===")
print(f"Best Vina score: {best_score_Stigmasterol} kcal/mol")
for i, e in enumerate(energies_Stigmasterol):
    print(f"  Pose {i+1}: {e[0]:.3f}")

vina_out_sdf_Stigmasterol = vina_out_pdbqt_Stigmasterol.replace('.pdbqt', '.sdf')
pdbqt_to_sdf(pdbqt_file=vina_out_pdbqt_Stigmasterol, output=vina_out_sdf_Stigmasterol)

# ---- If the poses collapse after 1-2, or the score is positive/near-zero
# (same failure signature seen against COX-2/5IKQ and 5-LOX/3O8Y before
# their fixes), uncomment and re-run this block ----
# STIGMASTEROL_EXHAUSTIVENESS = 32
# STIGMASTEROL_BOX_PAD = 8.0
# v_Stigmasterol = Vina(sf_name='vina', seed=VINA_SEED)
# v_Stigmasterol.set_receptor(RECEPTOR_PDBQT)
# v_Stigmasterol.set_ligand_from_file(lig_pdbqt_Stigmasterol)
# v_Stigmasterol.compute_vina_maps(
#     center=[center['center_x'], center['center_y'], center['center_z']],
#     box_size=[size['size_x'] + STIGMASTEROL_BOX_PAD,
#               size['size_y'] + STIGMASTEROL_BOX_PAD,
#               size['size_z'] + STIGMASTEROL_BOX_PAD])
# v_Stigmasterol.dock(exhaustiveness=STIGMASTEROL_EXHAUSTIVENESS, n_poses=10)
# v_Stigmasterol.write_poses(vina_out_pdbqt_Stigmasterol, n_poses=10, overwrite=True)
# energies_Stigmasterol = v_Stigmasterol.energies(n_poses=10)
# best_score_Stigmasterol = round(float(energies_Stigmasterol[0][0]), 3)
# print("=== Stigmasterol (exhaustiveness=32, widened box) ===")
# print(f"Best Vina score: {best_score_Stigmasterol} kcal/mol")
# for i, e in enumerate(energies_Stigmasterol):
#     print(f"  Pose {i+1}: {e[0]:.3f}")
# pdbqt_to_sdf(pdbqt_file=vina_out_pdbqt_Stigmasterol, output=vina_out_sdf_Stigmasterol)


  Stigmasterol: 20 conformers generated, lowest energy = 104.81 kcal/mol (conf 6)
=== Stigmasterol ===
Best Vina score: -9.03 kcal/mol
  Pose 1: -9.030
  Pose 2: -8.948
  Pose 3: -8.547
  Pose 4: -8.529
  Pose 5: -8.423
  Pose 6: -8.402
  Pose 7: -8.268
  Pose 8: -7.968
  Pose 9: -7.835
  Pose 10: -7.656

mode |   affinity | dist from best mode
     | (kcal/mol) | rmsd l.b.| rmsd u.b.
-----+------------+----------+----------
   1       -8.149          0          0
   2       -8.008      3.498      4.823
   3       -7.942      3.376      4.894
   4       -7.522      3.228      5.193
   5       -7.472      1.009      2.583
   6       -7.462      2.619      7.776
   7       -7.404      3.519      5.495
   8        -7.24      2.168      3.482
   9       -7.233      3.042      5.629
  10         -7.1      2.606      7.434
Computing Vina grid ... done.
Performing docking (random seed: 42) ... 
0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|---

### 7.4. Dock **7-O-Glucosyl-3-methylkaempferol** against Aldose Reductase (2FZD)

Flavonoid glycoside isolated from the aerial parts (butanolic extract) of *Lamium amplexicaule* by Laggoune et al. (2008); reported formula C22H22O11, structurally a 3-O-methyl ether of kaempferol-7-O-glucoside (cf. PubChem CID 5480982 for the unmethylated parent, C21H20O11).

In [15]:
# ---- 7-O-Glucosyl-3-methylkaempferol (Lamium_amplexicaule, Laggoune et al. 2008) ----
smiles_GlcMethylKaempferol = "COc1c(-c2ccc(O)cc2)oc2cc(O[C@@H]3O[C@H](CO)[C@@H](O)[C@H](O)[C@H]3O)cc(O)c2c1=O"
lig_pdbqt_GlcMethylKaempferol = prepare_ligand_from_smiles("GlcMethylKaempferol", smiles_GlcMethylKaempferol)

v_GlcMethylKaempferol = Vina(sf_name='vina', seed=VINA_SEED)
v_GlcMethylKaempferol.set_receptor(RECEPTOR_PDBQT)
v_GlcMethylKaempferol.set_ligand_from_file(lig_pdbqt_GlcMethylKaempferol)
v_GlcMethylKaempferol.compute_vina_maps(center=[center['center_x'], center['center_y'], center['center_z']],
                     box_size=[size['size_x'], size['size_y'], size['size_z']])

v_GlcMethylKaempferol.dock(exhaustiveness=10, n_poses=10)
vina_out_pdbqt_GlcMethylKaempferol = os.path.join(DOCKING_DIR, "GlcMethylKaempferol_2FZD_vina_out.pdbqt")
v_GlcMethylKaempferol.write_poses(vina_out_pdbqt_GlcMethylKaempferol, n_poses=10, overwrite=True)

energies_GlcMethylKaempferol = v_GlcMethylKaempferol.energies(n_poses=10)
best_score_GlcMethylKaempferol = round(float(energies_GlcMethylKaempferol[0][0]), 3)

print("=== GlcMethylKaempferol ===")
print(f"Best Vina score: {best_score_GlcMethylKaempferol} kcal/mol")
for i, e in enumerate(energies_GlcMethylKaempferol):
    print(f"  Pose {i+1}: {e[0]:.3f}")

vina_out_sdf_GlcMethylKaempferol = vina_out_pdbqt_GlcMethylKaempferol.replace('.pdbqt', '.sdf')
pdbqt_to_sdf(pdbqt_file=vina_out_pdbqt_GlcMethylKaempferol, output=vina_out_sdf_GlcMethylKaempferol)


  GlcMethylKaempferol: 20 conformers generated, lowest energy = 164.14 kcal/mol (conf 13)

mode |   affinity | dist from best mode
     | (kcal/mol) | rmsd l.b.| rmsd u.b.
-----+------------+----------+----------
   1        -9.03          0          0
   2       -8.948      1.316      2.604
   3       -8.547      4.791      8.051
   4       -8.529      1.473      3.265
   5       -8.423      4.684      8.524
   6       -8.402      1.656      3.156
   7       -8.268      2.606      4.924
   8       -7.968      2.451      5.001
   9       -7.835      4.808      9.595
  10       -7.656      2.484      5.524
Computing Vina grid ... done.
Performing docking (random seed: 42) ... 
0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************
=== GlcMethylKaempferol ===
Best Vina score: -8.373 kcal/mol
  Pose 1: -8.373
  Pose 2: -8.306
  Pose 3: -8.134
  Pose 4: -8.051
  Pose 5: -7.671
  Pose 6: 

### 7.5. Dock **Sitosterol** against Aldose Reductase (2FZD)

Phytosterol isolated from the roots (Soxhlet acetone extraction) of *Lamium amplexicaule* by Laggoune et al. (2008); C29H50O, structurally identical to stigmasterol above except for a saturated C22-C23 side-chain bond.

In [ ]:
# ---- Sitosterol (Lamium_amplexicaule, Laggoune et al. 2008) ----
smiles_Sitosterol = "CC[C@H](CC[C@@H](C)[C@H]1CC[C@@H]2[C@@]1(CC[C@H]3[C@H]2CC=C4[C@@]3(CC[C@@H](C4)O)C)C)C(C)C"
lig_pdbqt_Sitosterol = prepare_ligand_from_smiles("Sitosterol", smiles_Sitosterol)

v_Sitosterol = Vina(sf_name='vina', seed=VINA_SEED)
v_Sitosterol.set_receptor(RECEPTOR_PDBQT)
v_Sitosterol.set_ligand_from_file(lig_pdbqt_Sitosterol)
v_Sitosterol.compute_vina_maps(center=[center['center_x'], center['center_y'], center['center_z']],
                     box_size=[size['size_x'], size['size_y'], size['size_z']])

v_Sitosterol.dock(exhaustiveness=10, n_poses=10)
vina_out_pdbqt_Sitosterol = os.path.join(DOCKING_DIR, "Sitosterol_2FZD_vina_out.pdbqt")
v_Sitosterol.write_poses(vina_out_pdbqt_Sitosterol, n_poses=10, overwrite=True)

energies_Sitosterol = v_Sitosterol.energies(n_poses=10)
best_score_Sitosterol = round(float(energies_Sitosterol[0][0]), 3)

print("=== Sitosterol ===")
print(f"Best Vina score: {best_score_Sitosterol} kcal/mol")
for i, e in enumerate(energies_Sitosterol):
    print(f"  Pose {i+1}: {e[0]:.3f}")

vina_out_sdf_Sitosterol = vina_out_pdbqt_Sitosterol.replace('.pdbqt', '.sdf')
pdbqt_to_sdf(pdbqt_file=vina_out_pdbqt_Sitosterol, output=vina_out_sdf_Sitosterol)

# ---- If poses collapse after 1-2 (as seen with Stigmasterol in this / other targets), uncomment ----
# STEROL_EXHAUSTIVENESS = 32
# STEROL_BOX_PAD = 6.0
# v_Sitosterol = Vina(sf_name='vina', seed=VINA_SEED)
# v_Sitosterol.set_receptor(RECEPTOR_PDBQT)
# v_Sitosterol.set_ligand_from_file(lig_pdbqt_Sitosterol)
# v_Sitosterol.compute_vina_maps(
#     center=[center['center_x'], center['center_y'], center['center_z']],
#     box_size=[size['size_x'] + STEROL_BOX_PAD,
#               size['size_y'] + STEROL_BOX_PAD,
#               size['size_z'] + STEROL_BOX_PAD])
# v_Sitosterol.dock(exhaustiveness=STEROL_EXHAUSTIVENESS, n_poses=10)
# v_Sitosterol.write_poses(vina_out_pdbqt_Sitosterol, n_poses=10, overwrite=True)
# energies_Sitosterol = v_Sitosterol.energies(n_poses=10)
# best_score_Sitosterol = round(float(energies_Sitosterol[0][0]), 3)
# pdbqt_to_sdf(pdbqt_file=vina_out_pdbqt_Sitosterol, output=vina_out_sdf_Sitosterol)


  Sitosterol: 20 conformers generated, lowest energy = 103.17 kcal/mol (conf 6)


### 7.4. Total docking results table -- Lamium_amplexicaule vs Aldose Reductase (2FZD)

In [ ]:
docking_rows_amplexicaule_AR = [
    {"Compound": "Chrysoeriol",  "Target": "Aldose_reductase (2FZD)", "Best_Vina_Score_kcal_mol": best_score_Chrysoeriol},
    {"Compound": "Stigmasterol", "Target": "Aldose_reductase (2FZD)", "Best_Vina_Score_kcal_mol": best_score_Stigmasterol},
    {"Compound": "GlcMethylKaempferol", "Target": "Aldose_reductase (2FZD)", "Best_Vina_Score_kcal_mol": best_score_GlcMethylKaempferol},
    {"Compound": "Sitosterol",          "Target": "Aldose_reductase (2FZD)", "Best_Vina_Score_kcal_mol": best_score_Sitosterol},
]

docking_df_amplexicaule_AR = pd.DataFrame(docking_rows_amplexicaule_AR).sort_values("Best_Vina_Score_kcal_mol").reset_index(drop=True)
docking_df_amplexicaule_AR.to_csv(os.path.join(DOCKING_DIR, "lamium_amplexicaule_vs_AldoseReductase_vina.csv"), index=False)
docking_df_amplexicaule_AR


### 7.5. Figure -- docking score comparison, Lamium_amplexicaule vs Aldose Reductase

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3))
colors = ['#7b2cbf' if s == docking_df_amplexicaule_AR['Best_Vina_Score_kcal_mol'].min() else '#c8b6ff'
          for s in docking_df_amplexicaule_AR['Best_Vina_Score_kcal_mol']]
ax.barh(docking_df_amplexicaule_AR['Compound'], docking_df_amplexicaule_AR['Best_Vina_Score_kcal_mol'], color=colors)
ax.set_xlabel('Best Vina docking score (kcal/mol)')
ax.set_title('Lamium amplexicaule compounds vs. Aldose Reductase (2FZD)')
ax.invert_yaxis()
for i, v in enumerate(docking_df_amplexicaule_AR['Best_Vina_Score_kcal_mol']):
    ax.text(v - 0.15, i, f"{v:.2f}", va='center', ha='right', fontsize=9, color='white')
plt.tight_layout()

fig_path_amplexicaule_AR = os.path.join(FIGURES_DIR, 'lamium_amplexicaule_vs_AldoseReductase_scores.png')
plt.savefig(fig_path_amplexicaule_AR, dpi=300, bbox_inches='tight')
plt.show()
print("Saved figure:", fig_path_amplexicaule_AR)


### 7.6. ProLif interaction fingerprints -- Lamium_amplexicaule vs Aldose Reductase (each compound, own cell)

`n_jobs=1` avoids a multiprocessing/pickling crash in this environment;
`to_dataframe()` takes no arguments in ProLif 2.2.1.

In [ ]:
prot_fp = mda.Universe("2FZD_clean_H_fix.pdb")  # fix_protein() output
prot_fp = plf.Molecule.from_mda(prot_fp, NoImplicit=False)
print(type(prot_fp))
print("Chains in prot_fp:", {r.chain for r in prot_fp.residues})

In [ ]:
# ---- Helper: render a ProLIF LigNetwork widget to a real PNG file ----
import os
import sys
import subprocess

_RENDER_WORKER_PATH = os.path.join(os.getcwd(), "_lignetwork_render_worker.py")

_RENDER_WORKER_SOURCE = '''
import sys
from playwright.sync_api import sync_playwright

html_path, png_path, width, height, wait_ms = sys.argv[1:6]
width, height, wait_ms = int(width), int(height), int(wait_ms)

with sync_playwright() as p:
    browser = p.chromium.launch()
    page = browser.new_page(viewport={"width": width, "height": height})
    page.goto(f"file://{html_path}")
    page.wait_for_timeout(wait_ms)
    try:
        page.evaluate("if (typeof network !== 'undefined') { network.fit({animation:false}); }")
        page.wait_for_timeout(300)
    except Exception:
        pass
    page.screenshot(path=png_path)
    browser.close()
'''

def lignetwork_to_png(net, png_path, width=1000, height=800, wait_ms=1500):
    """Render a ProLIF LigNetwork object to a standalone PNG file."""
    with open(_RENDER_WORKER_PATH, "w") as f:
        f.write(_RENDER_WORKER_SOURCE)

    tmp_html = png_path.rsplit(".", 1)[0] + "_tmp.html"
    net.save(tmp_html)

    result = subprocess.run(
        [sys.executable, _RENDER_WORKER_PATH, os.path.abspath(tmp_html),
         os.path.abspath(png_path), str(width), str(height), str(wait_ms)],
        capture_output=True, text=True,
    )

    if result.returncode != 0:
        print("Render worker STDOUT:", result.stdout)
        print("Render worker STDERR:", result.stderr)
        raise RuntimeError(f"lignetwork_to_png failed for {png_path}")

    os.remove(tmp_html)
    print("Saved PNG:", png_path)

In [ ]:
# ---- ProLif: Chrysoeriol ----
lig_suppl_Chrysoeriol = list(plf.sdf_supplier(vina_out_sdf_Chrysoeriol))
fp_Chrysoeriol = plf.Fingerprint()
fp_Chrysoeriol.run_from_iterable(lig_suppl_Chrysoeriol, prot_fp, n_jobs=1)
results_df_Chrysoeriol = fp_Chrysoeriol.to_dataframe()
print("=== Chrysoeriol ===")
display(results_df_Chrysoeriol)

valid_frames = results_df_Chrysoeriol[results_df_Chrysoeriol.any(axis=1)].index
if len(valid_frames) == 0:
    print("No interactions detected in any pose for Chrysoeriol - skipping network plot.")
else:
    frame_to_use = valid_frames[0]
    print("Using frame:", frame_to_use)
    net_Chrysoeriol = fp_Chrysoeriol.plot_lignetwork(lig_suppl_Chrysoeriol[0], kind="frame", frame=frame_to_use, rotation=270)
    net_png_Chrysoeriol = os.path.join(FIGURES_DIR, "Chrysoeriol_2FZD_2Dnetwork.png")
    lignetwork_to_png(net_Chrysoeriol, net_png_Chrysoeriol)

In [ ]:
# ---- ProLif: Stigmasterol ----
lig_suppl_Stigmasterol = list(plf.sdf_supplier(vina_out_sdf_Stigmasterol))
fp_Stigmasterol = plf.Fingerprint()
fp_Stigmasterol.run_from_iterable(lig_suppl_Stigmasterol, prot_fp, n_jobs=1)
results_df_Stigmasterol = fp_Stigmasterol.to_dataframe()
print("=== Stigmasterol ===")
display(results_df_Stigmasterol)

valid_frames = results_df_Stigmasterol[results_df_Stigmasterol.any(axis=1)].index
if len(valid_frames) == 0:
    print("No interactions detected in any pose for Stigmasterol - skipping network plot.")
else:
    frame_to_use = valid_frames[0]
    print("Using frame:", frame_to_use)
    net_Stigmasterol = fp_Stigmasterol.plot_lignetwork(lig_suppl_Stigmasterol[0], kind="frame", frame=frame_to_use, rotation=270)
    net_png_Stigmasterol = os.path.join(FIGURES_DIR, "Stigmasterol_2FZD_2Dnetwork.png")
    lignetwork_to_png(net_Stigmasterol, net_png_Stigmasterol)

In [ ]:
# ---- ProLif: GlcMethylKaempferol ----
lig_suppl_GlcMethylKaempferol = list(plf.sdf_supplier(vina_out_sdf_GlcMethylKaempferol))
fp_GlcMethylKaempferol = plf.Fingerprint()
fp_GlcMethylKaempferol.run_from_iterable(lig_suppl_GlcMethylKaempferol, prot_fp, n_jobs=1)
results_df_GlcMethylKaempferol = fp_GlcMethylKaempferol.to_dataframe()
print("=== GlcMethylKaempferol ===")
display(results_df_GlcMethylKaempferol)

valid_frames = results_df_GlcMethylKaempferol[results_df_GlcMethylKaempferol.any(axis=1)].index
if len(valid_frames) == 0:
    print("No interactions detected in any pose for GlcMethylKaempferol - skipping network plot.")
else:
    frame_to_use = valid_frames[0]
    print("Using frame:", frame_to_use)
    net_GlcMethylKaempferol = fp_GlcMethylKaempferol.plot_lignetwork(lig_suppl_GlcMethylKaempferol[0], kind="frame", frame=frame_to_use, rotation=270)
    net_png_GlcMethylKaempferol = os.path.join(FIGURES_DIR, "GlcMethylKaempferol_2FZD_2Dnetwork.png")
    lignetwork_to_png(net_GlcMethylKaempferol, net_png_GlcMethylKaempferol)

In [ ]:
# ---- ProLif: Sitosterol ----
lig_suppl_Sitosterol = list(plf.sdf_supplier(vina_out_sdf_Sitosterol))
fp_Sitosterol = plf.Fingerprint()
fp_Sitosterol.run_from_iterable(lig_suppl_Sitosterol, prot_fp, n_jobs=1)
results_df_Sitosterol = fp_Sitosterol.to_dataframe()
print("=== Sitosterol ===")
display(results_df_Sitosterol)

valid_frames = results_df_Sitosterol[results_df_Sitosterol.any(axis=1)].index
if len(valid_frames) == 0:
    print("No interactions detected in any pose for Sitosterol - skipping network plot.")
else:
    frame_to_use = valid_frames[0]
    print("Using frame:", frame_to_use)
    net_Sitosterol = fp_Sitosterol.plot_lignetwork(lig_suppl_Sitosterol[0], kind="frame", frame=frame_to_use, rotation=270)
    net_png_Sitosterol = os.path.join(FIGURES_DIR, "Sitosterol_2FZD_2Dnetwork.png")
    lignetwork_to_png(net_Sitosterol, net_png_Sitosterol)

### 7.7. Figure -- best-scoring compound vs. the real native ligand (tolrestat)

In [ ]:
best_row_amplexicaule_AR = docking_df_amplexicaule_AR.iloc[0]
best_compound_amplexicaule_AR = best_row_amplexicaule_AR['Compound']
print("Best-scoring compound:", best_compound_amplexicaule_AR, "|", best_row_amplexicaule_AR['Best_Vina_Score_kcal_mol'], "kcal/mol")

best_sdf_amplexicaule_AR = {
    "Chrysoeriol": vina_out_sdf_Chrysoeriol,
    "Stigmasterol": vina_out_sdf_Stigmasterol,
    "GlcMethylKaempferol": vina_out_sdf_GlcMethylKaempferol,
    "Sitosterol":          vina_out_sdf_Sitosterol,
}[best_compound_amplexicaule_AR]

view = py3Dmol.view()
view.removeAllModels()
view.setViewStyle({'style':'outline','color':'black','width':0.1})

view.addModel(open(RECEPTOR_PDB,'r').read(), format='pdb')
Prot = view.getModel()
Prot.setStyle({'cartoon':{'arrows':True, 'tubes':True, 'style':'oval', 'color':'white'}})
view.addSurface(py3Dmol.VDW, {'opacity':0.6,'color':'white'})

view.addModel(open(NATIVE_LIG,'r').read(), format='mol2')
ref_m = view.getModel()
ref_m.setStyle({}, {'stick':{'colorscheme':'magentaCarbon','radius':0.2}})

results_amp_AR = Chem.SDMolSupplier(best_sdf_amplexicaule_AR)
p_amp_AR = Chem.MolToMolBlock(results_amp_AR[0], False)
print('Reference (tolrestat): Magenta | Best compound pose: Cyan')
print('Pose: {} | Score: {}'.format(results_amp_AR[0].GetProp('Pose'), results_amp_AR[0].GetProp('Score')))

view.addModel(p_amp_AR, 'mol')
x = view.getModel()
x.setStyle({}, {'stick':{'colorscheme':'cyanCarbon','radius':0.2}})

view.zoomTo()
view.show()
